**German Credit Dataset**

# 03.2 - Fairness-Aware Models Pipeline (Revisado)

**Objectives**
- Build and run fairness-aware ML pipelines using AIF360 in-processing models
- Train and evaluate PrejudiceRemover and AdversarialDebiasing automatically
- Export performance and fairness results compatible with the notebook 04 analysis


In [ ]:
import os
import time
import sys
import importlib
import random

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
import tensorflow.compat.v1 as tf
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.inprocessing import AdversarialDebiasing, PrejudiceRemover


In [ ]:
sys.path.append('utils')

os.environ['PYTHONHASHSEED'] = '2'
random.seed(2)
np.random.seed(2)

In [ ]:
import evaluation

importlib.reload(evaluation)

## 1. Load Data

In [ ]:
file_path = '../data/processed/german_df_processed_2.csv'
df = pd.read_csv(file_path, sep=r',', header=0)

In [ ]:
df

## 2. Setup: Key Variables, Helper Functions, and Pipeline Construction

### 2.1. Key Variables

In [ ]:
NUMERIC_COLS = [
    'duration_months', 'credit_amount'
]

NOMINAL_COLS = [
    'sex', 'age_cat', 'checking_account_status', 'credit_history',
    'purpose', 'savings_account_status', 'employment_status',
    'guarantors', 'property', 'other_debts', 'housing', 'job',
    'own_telephone?', 'foreign_worker?','dependents'
]

ORDINAL_COLS = [
    'installment_rate', 'residence_duration', 'existing_credits_count'
]

CATEGORICAL_COLS = NOMINAL_COLS + ORDINAL_COLS

In [ ]:
TARGET_COLUMN = 'good_client?'
FAVORABLE_LABEL = 1   # 'Good client'
UNFAVORABLE_LABEL = 0 # 'No good client'

SENSITIVE_ATTR = 'sex'
PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Male
UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # Female

# SENSITIVE_ATTR = 'foreign_worker?'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Foreign
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 0}] # Local

# SENSITIVE_ATTR = 'age_cat'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # age >= 25
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # age < 25 

### 2.2. Helper Functions

In [ ]:
def df_to_aif360(df_input):
    return BinaryLabelDataset(
        df=df_input,
        label_names=[TARGET_COLUMN],
        protected_attribute_names=[SENSITIVE_ATTR],
        favorable_label=FAVORABLE_LABEL,
        unfavorable_label=UNFAVORABLE_LABEL
    )


In [ ]:
def encode_features_for_inprocessing(ds_train, ds_test):
    """
    Encodes/scales all features except SENSITIVE_ATTR, which must keep its
    original values since BinaryLabelDataset derives `protected_attributes`
    from `features` by column index.

    Args:
        ds_train (BinaryLabelDataset): Training dataset.
        ds_test  (BinaryLabelDataset): Test dataset.

    Returns:
        tuple[BinaryLabelDataset, BinaryLabelDataset]: datasets with encoded/
            scaled features, except for the sensitive attribute.
    """
    feature_names = ds_train.feature_names

    try:
        sensitive_idx = feature_names.index(SENSITIVE_ATTR)
    except ValueError:
        raise ValueError(
            f"SENSITIVE_ATTR='{SENSITIVE_ATTR}' not found in feature_names={feature_names}."
        )

    cols_to_transform = [i for i in range(len(feature_names)) if i != sensitive_idx]

    X_train = ds_train.features.copy()
    X_test  = ds_test.features.copy()

    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train[:, cols_to_transform] = enc.fit_transform(X_train[:, cols_to_transform])
    X_test[:, cols_to_transform]  = enc.transform(X_test[:, cols_to_transform])

    scaler = StandardScaler()
    X_train[:, cols_to_transform] = scaler.fit_transform(X_train[:, cols_to_transform])
    X_test[:, cols_to_transform]  = scaler.transform(X_test[:, cols_to_transform])

    ds_train_enc = ds_train.copy()
    ds_test_enc  = ds_test.copy()

    ds_train_enc.features = X_train
    ds_test_enc.features  = X_test

    return ds_train_enc, ds_test_enc

In [ ]:
def summarize_folds_to_df_summary(config_id, n_splits, df_folds):
    """
    Aggregates metrics from multiple folds into a single summary row with mean/std.

    Args:
        config_id (str): Configuration identifier.
        n_splits (int): Number of folds.
        df_folds (pd.DataFrame): DataFrame with one row per fold.

    Returns:
        pd.DataFrame: Single row with mean_* and std_* columns for each metric.
    """
    metric_cols = [c for c in df_folds.columns
                   if c not in ['config_id', 'fold', 'Pipeline']]

    df_mean = df_folds[metric_cols].mean(numeric_only=True).to_frame().T
    df_std  = df_folds[metric_cols].std(numeric_only=True).to_frame().T

    df_summary = df_mean.add_prefix('mean_').join(df_std.add_prefix('std_'))
    df_summary.insert(0, 'config_id', config_id)
    df_summary.insert(1, 'n_splits',  n_splits)

    return df_summary


### 2.3. Pipeline Construction

In [ ]:
def run_cv_aif360_inprocessing(
    model_name, df,
    n_splits=5,
    random_state=2,
    shuffle=True,
    **model_kwargs
):
    """
    Cross-validation (StratifiedKFold) for an AIF360 in-processing model.

    Args:
        model_name (str): 'prejudice_remover' or 'adversarial_debiasing'.
        df (pd.DataFrame): Full dataset containing TARGET_COLUMN and SENSITIVE_ATTR.
        n_splits (int): Number of folds.
        random_state (int): Seed for StratifiedKFold.
        shuffle (bool): Whether to shuffle before creating the folds.
        **model_kwargs: Model-specific hyperparameters.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: (df_folds, df_summary)
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=shuffle,
                          random_state=random_state)
    y = df[TARGET_COLUMN].values

    config_id = (
        f'AIF360|model={model_name}'
        f'|sensitive={SENSITIVE_ATTR}'
        f'|encoder=ordinal_excl_sensitive'
        f'|scaler=standardization'
        f'|{model_kwargs}'
    ).replace(' ', '')

    print(f'Running CV: {config_id} | n_splits={n_splits}')

    fold_rows = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(df, y), start=1):
        print(f'\n[Fold {fold}/{n_splits}] Training and evaluating...')

        df_train = df.iloc[train_idx].copy()
        df_test  = df.iloc[test_idx].copy()

        # 1) Convert to BinaryLabelDataset
        ds_train = df_to_aif360(df_train)
        ds_test  = df_to_aif360(df_test)

        # 2) Encode + scale (excludes SENSITIVE_ATTR)
        ds_train_enc, ds_test_enc = encode_features_for_inprocessing(
            ds_train, ds_test
        )

        # 3) Train the in-processing model
        if model_name == 'prejudice_remover':
            # PrejudiceRemover (Kamiran & Calders, 2010): penalizes the mutual
            # information between predictions and the sensitive attribute.
            # eta=0 -> plain regularized logistic regression (no fairness)
            # eta=25 -> original paper's default
            # eta>50 -> strong fairness push, may hurt performance significantly
            model = PrejudiceRemover(
                sensitive_attr=SENSITIVE_ATTR,
                **model_kwargs
            )
            model.fit(ds_train_enc)
            ds_pred = model.predict(ds_test_enc)

        elif model_name == 'adversarial_debiasing':
            # AdversarialDebiasing (Zhang et al., 2018): trains a classifier
            # and an adversary in a minmax setup. The seed must be set AFTER
            # reset_default_graph() and BEFORE building the model (which
            # constructs the TF graph), since reset_default_graph() clears
            # any seed set outside the loop.
            tf.reset_default_graph()
            tf.random.set_random_seed(2)
            sess = tf.Session()

            model = AdversarialDebiasing(
                privileged_groups=PRIVILEGED_GROUPS,
                unprivileged_groups=UNPRIVILEGED_GROUPS,
                scope_name=f'adv_debias_fold_{fold}',
                sess=sess,
                **model_kwargs
            )
            model.fit(ds_train_enc)
            ds_pred = model.predict(ds_test_enc)
            sess.close()

        else:
            raise ValueError(
                f"model_name='{model_name}' is invalid. "
                "Use 'prejudice_remover' or 'adversarial_debiasing'."
            )

        # 4) Evaluate performance and fairness
        df_metrics = evaluation.evaluate_pipeline(
            ds_test_enc, ds_pred,
            UNPRIVILEGED_GROUPS, PRIVILEGED_GROUPS,
            pipeline_name=f'{config_id}|fold={fold}'
        )

        df_metrics = df_metrics.copy()
        df_metrics.insert(0, 'fold',      fold)
        df_metrics.insert(0, 'config_id', config_id)
        fold_rows.append(df_metrics)

    # Aggregate results
    df_folds   = pd.concat(fold_rows, ignore_index=True)
    df_summary = summarize_folds_to_df_summary(config_id, n_splits, df_folds)

    return df_folds, df_summary

In [ ]:
MODEL_PREFIX = {
    "prejudice_remover": "PR",
    "adversarial_debiasing": "AD"
}


def _assign_model_id(results):
    """Adds a compact 'model_id' column (e.g. PR001, AD002...), ranked per model prefix."""
    prefix = results['model'].map(MODEL_PREFIX).fillna('MD')
    rank = prefix.groupby(prefix).cumcount() + 1
    results = results.copy()
    results['model_id'] = prefix + rank.astype(str).str.zfill(3)
    return results


def _reorder_results_columns(results):
    """Identifiers/config columns first, then metrics, then everything else."""
    front = [
        'model_id', 'model', 'source', 'scaler', 'encoder', 'bias_pre',
        'tuning', 'bias_post', 'n_splits',
        'eta', 'adversary_loss_weight', 'num_epochs',
        'classifier_num_hidden_units', 'debias',
    ]
    front = [c for c in front if c in results.columns]
    metrics = [c for c in results.columns if c.startswith('mean_') or c.startswith('std_')]
    rest = [c for c in results.columns if c not in front + metrics]
    return results[front + metrics + rest]


def run_param_sweep(model_name, param_list, df, n_splits=3):
    """
    Runs run_cv_aif360_inprocessing for multiple hyperparameter configurations
    and returns a consolidated DataFrame with the results of all configurations,
    already formatted with the same column contract as the classic pipeline
    results (model_id, model, source, scaler, encoder, bias_pre, tuning,
    bias_post, n_splits, mean_*/std_* metrics), so it can be concatenated
    directly with the classic results without any post-hoc parsing.

    Handles errors per configuration: individual failures are logged without
    interrupting the remaining runs.

    Args:
        model_name (str): 'prejudice_remover' or 'adversarial_debiasing'.
        param_list (list[dict]): List of hyperparameter dictionaries.
        df (pd.DataFrame): Full dataset.
        n_splits (int): Number of CV folds.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: (results_df, errors_df).
    """
    summaries = []
    errors    = []
    total     = len(param_list)
    start_all = time.time()

    for i, params in enumerate(param_list, start=1):
        print(f'\n[{i}/{total}] {model_name} | params={params}')

        try:
            _, summary_df = run_cv_aif360_inprocessing(
                model_name, df=df, n_splits=n_splits, **params
            )
            summary_df = summary_df.copy()

            # Store params as columns for easier downstream analysis
            for k, v in params.items():
                summary_df[k] = v

            # Structured config columns, matching the classic pipeline contract.
            # In-processing models don't go through separate bias_pre/tuning/
            # bias_post pipeline stages -- the model itself is the mitigation,
            # so these are fixed to 'none'.
            summary_df['model']      = model_name
            summary_df['source']     = 'fairness'
            summary_df['scaler']     = 'standardization'
            summary_df['encoder']    = 'ordinal_excl_sensitive'
            summary_df['bias_pre']   = 'none'
            summary_df['tuning']     = 'none'
            summary_df['bias_post']  = 'none'
            summary_df['n_splits']   = n_splits

            summaries.append(summary_df)

        except Exception as e:
            print(f'  [ERROR] {type(e).__name__}: {e}')
            errors.append({
                'model_name': model_name,
                'params':     str(params),
                'error_type': type(e).__name__,
                'error_msg':  str(e)
            })

    total_time = time.time() - start_all
    print(f'\n{"#"*70}')
    print(f'{model_name} sweep completed in {total_time/60:.1f} min')
    print(f'Success: {len(summaries)} | Failed: {len(errors)}')
    print(f'{"#"*70}')

    if summaries:
        results_df = pd.concat(summaries, ignore_index=True)
        results_df = _assign_model_id(results_df)
        results_df = _reorder_results_columns(results_df)
    else:
        results_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)
    return results_df, errors_df

## 3. Pipeline Execution

### 3.1. Single Model Test


#### 3.1.1. PrejudiceRemover

In [ ]:
print('\n=== PrejudiceRemover — Default Paper (eta=25) ===')
_, summary_pr_default = run_cv_aif360_inprocessing(
    'prejudice_remover', df=df, n_splits=3,
    eta=25.0 
)
display(summary_pr_default)


#### 3.1.2. AdversarialDebiasing

In [ ]:
tf.disable_eager_execution()

print('=== AdversarialDebiasing — Baseline (debias=False) ===')
_, summary_ad_baseline = run_cv_aif360_inprocessing(
    'adversarial_debiasing', df=df, n_splits=3,
    num_epochs=100,                  
    classifier_num_hidden_units=128,
    debias=False,                      
    adversary_loss_weight=0.1
)
display(summary_ad_baseline)

### 3.2. All models


#### 3.2.1. PrejudiceRemover — Parameter Sweep

In [ ]:
pr_param_list = [
    {'eta': eta}
    for eta in [0.0, 1.0, 5.0, 10.0, 25.0, 50.0, 100.0]
]

results_pr, errors_pr = run_param_sweep(
    'prejudice_remover', pr_param_list, df=df, n_splits=3
)


In [ ]:
if not results_pr.empty:
    cols_exibir = [
        'config_id', 'eta',
        'mean_F1-Score', 'std_F1-Score',
        'mean_Demographic Parity Diff.', 'std_Demographic Parity Diff.',
        'mean_Equal Opportunity Diff.', 'std_Equal Opportunity Diff.',
        'mean_Average Odds Diff.', 'std_Average Odds Diff.'
    ]
    cols_exibir = [c for c in cols_exibir if c in results_pr.columns]
    display(results_pr[cols_exibir].sort_values('eta', ascending=False))

if not errors_pr.empty:
    print('\nErros:')
    display(errors_pr)


In [ ]:
errors_pr

#### 3.2.2. AdversarialDebiasing — Parameter Sweep

In [ ]:
ad_param_list = [
    # ==========================================
    {'debias': False, 'num_epochs': 100, 'classifier_num_hidden_units': 512,  'adversary_loss_weight': 0},

    {'debias': True, 'num_epochs': 100, 'classifier_num_hidden_units': 512,  'adversary_loss_weight': 0.01},
    {'debias': True, 'num_epochs': 100, 'classifier_num_hidden_units': 512,  'adversary_loss_weight': 0.05},
    {'debias': True, 'num_epochs': 100, 'classifier_num_hidden_units': 512,  'adversary_loss_weight': 0.1},
    {'debias': True, 'num_epochs': 100, 'classifier_num_hidden_units': 512,  'adversary_loss_weight': 0.2},
    {'debias': True, 'num_epochs': 100, 'classifier_num_hidden_units': 512,  'adversary_loss_weight': 0.5},
    {'debias': True, 'num_epochs': 100, 'classifier_num_hidden_units': 512,  'adversary_loss_weight': 0.8},

]


results_ad, errors_ad = run_param_sweep(
    'adversarial_debiasing', ad_param_list, df=df, n_splits=3
)


In [ ]:
if not results_ad.empty:
    cols_exibir = [
        'config_id', 'debias', 'num_epochs', 
        'classifier_num_hidden_units', 'adversary_loss_weight',
        'mean_F1-Score', 'std_F1-Score',
        'mean_Demographic Parity Diff.', 'std_Demographic Parity Diff.',
        'mean_Equal Opportunity Diff.', 'std_Equal Opportunity Diff.',
        'mean_Average Odds Diff.', 'std_Average Odds Diff.'
    ]
    cols_exibir = [c for c in cols_exibir if c in results_ad.columns]
    display(results_ad[cols_exibir].sort_values('adversary_loss_weight', ascending=False))

if not errors_ad.empty:
    print('\nErros:')
    display(errors_ad)


In [ ]:
results_ad.head(40)

In [ ]:
errors_ad

## 4. Exporting Results

In [ ]:
all_results = pd.concat(
    [df for df in [results_pr, results_ad] if not df.empty],
    ignore_index=True
)

In [ ]:
file_out_path = '../data/results'

all_results.to_csv(
    file_out_path + '/fairness_models_results_sex.csv',
    index=False
)

print('Exportado: fairness_models_results_revised.csv')
print(f'Linhas: {len(all_results)} | Colunas: {len(all_results.columns)}')
